# Traveling salesperson (4 cities)

A salesperson leaves the **depot** (city 0), visits harbor, market and
tower once each, and returns.  City 0 is pinned at slot 0, so the
remaining $3\times 3$ one-hot matrix gives 9 binary variables.

The cost of a valid tour is its closed length.  Invalid (not one-hot)
bitstrings get a flat penalty.  We solve this with D-Wave Ocean's
simulated annealer.

This notebook is self-contained. It does not import `tsp.py`.

In [ ]:
import itertools

import dimod
import matplotlib.pyplot as plt
import neal
import numpy as np

In [ ]:
NAMES = ("depot", "harbor", "market", "tower")
DIST = np.array([
    [0.0, 2.0, 3.0, 2.5],
    [2.0, 0.0, 1.5, 4.0],
    [3.0, 1.5, 0.0, 1.0],
    [2.5, 4.0, 1.0, 0.0],
])
N_FREE = 3
N = N_FREE * N_FREE
PENALTY = 8.0


def idx(city, slot):
    return (city - 1) * N_FREE + (slot - 1)


def length(order):
    return sum(DIST[a, b] for a, b in zip(order, order[1:] + order[:1]))


def decode(bits):
    flags = [int(b) for b in bits[::-1]]
    slots = [0, -1, -1, -1]
    used_c, used_t = set(), set()
    for city in (1, 2, 3):
        ones = [t for t in (1, 2, 3) if flags[idx(city, t)] == 1]
        if len(ones) != 1:
            return None
        t = ones[0]
        if t in used_t:
            return None
        slots[t] = city
        used_t.add(t)
        used_c.add(city)
    return tuple(slots)

## Classical baseline

Enumerate all $3! = 6$ permutations of the three free cities.

In [ ]:
best, best_len = None, float("inf")
for tail in itertools.permutations((1, 2, 3)):
    tour = (0, *tail)
    L = length(tour)
    print(f"  {tour}  {[NAMES[i] for i in tour]}  length={L:.1f}")
    if L < best_len:
        best, best_len = tour, L
print(f"\nOptimal: {best}  {[NAMES[i] for i in best]}  length={best_len:.1f}")

## QUBO formulation

Variables: $x_{c,s} \in \{0,1\}$ for city $c \in \{1,2,3\}$, slot
$s \in \{1,2,3\}$. City 0 is fixed at slot 0.

Penalties:
- Each city in exactly one slot (row constraint)
- Each slot holds exactly one city (column constraint)

Objective: minimise tour length $\sum_{s} \text{DIST}[\text{city}(s), \text{city}(s+1)]$.

In [ ]:
def build_qubo():
    bqm = dimod.BinaryQuadraticModel("BINARY")
    names = [f"x_{c}_{s}" for c in range(1, 4) for s in range(1, 4)]
    for n in names:
        bqm.add_variable(n, 0.0)

    for c in range(1, 4):
        for s1 in range(1, 4):
            for s2 in range(s1 + 1, 4):
                bqm.add_interaction(f"x_{c}_{s1}", f"x_{c}_{s2}", PENALTY)

    for s in range(1, 4):
        for c1 in range(1, 4):
            for c2 in range(c1 + 1, 4):
                bqm.add_interaction(f"x_{c1}_{s}", f"x_{c2}_{s}", PENALTY)

    for c in range(1, 4):
        for s in range(1, 4):
            bqm.add_variable(f"x_{c}_{s}", -PENALTY)

    prev_slot = 0
    for slot in range(1, 4):
        for c_prev in range(1, 4):
            v_prev = f"x_{c_prev}_{prev_slot}" if prev_slot > 0 else None
            for c_next in range(1, 4):
                cost = DIST[c_prev, c_next]
                if v_prev is not None:
                    bqm.add_interaction(v_prev, f"x_{c_next}_{slot}", cost)
                else:
                    bqm.add_variable(f"x_{c_next}_{slot}", cost)
        prev_slot = slot

    last_slot = N_FREE
    for c_last in range(1, 4):
        for c_return in range(1, 4):
            bqm.add_interaction(
                f"x_{c_last}_{last_slot}", f"x_{c_return}_{1}",
                DIST[c_last, c_return],
            )
    return bqm


bqm = build_qubo()
print(f"Variables:   {len(bqm.variables)}")
print(f"Interactions: {len(bqm.quadratic)}")

## Solve with simulated annealing

In [ ]:
sampler = neal.SimulatedAnnealingSampler()
response = sampler.sample(bqm, num_reads=500, num_sweeps=1000)

sample = response.first.sample
bits = "".join(str(int(round(sample.get(v, 0)))) for v in bqm.variables)
tour = decode(bits)

print(f"Raw bits:  {bits}")
print(f"Decoded:   {tour}")
if tour is not None:
    cost = length(tour)
    print(f"Route:     {' -> '.join(NAMES[i] for i in tour)}")
    print(f"Total cost: {cost:.1f}")
else:
    print("Invalid tour — increase num_reads or PENALTY")

## Visualise the route

In [ ]:
if tour is not None:
    coords = np.array([[0.0, 1.0], [1.0, 1.0], [1.0, 0.0], [0.0, 0.0]])
    fig, ax = plt.subplots(figsize=(5, 5))
    for k in range(len(tour)):
        i, j = tour[k], tour[(k + 1) % len(tour)]
        ax.annotate("", xy=coords[j], xytext=coords[i],
                     arrowprops=dict(arrowstyle="-|>", color="C0", lw=2))
    for i in range(4):
        c = "C3" if i == 0 else "C0"
        ax.plot(*coords[i], "o", color=c, markersize=14, zorder=5)
        ax.annotate(NAMES[i], coords[i], textcoords="offset points",
                     xytext=(10, -5), fontsize=11, fontweight="bold")
    ax.set_title(f"TSP  route: {' -> '.join(NAMES[i] for i in tour)}\ntotal length = {cost:.1f}")
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    plt.show()